In [2]:
# ============================================================
# STEP 1 — imports + connect to database
# ============================================================
import pandas as pd
import geopandas as gpd
import sqlite3
import numpy as np




db_Path = r"C:\cabarrus-gis-prep\sql\databases\cabarrus_recreation.db"
conn = sqlite3.connect(db_Path)

#HERE IM SKIPPING THE creation of databases because the park data is already a table in the db_path. But usually you would need to 
#pd.read_csv() the data and then use to_sql() to put it in the database. Check park_analysisPractice that where I did that.



print(pd.read_sql_query("SELECT * FROM parks LIMIT 5",conn))

              X              Y  OBJECTID FACILITYID  \
0  1.515174e+06  644107.563114         1      FAC-1   
1  1.517086e+06  590589.513982         2      FAC-6   
2  1.572810e+06  603840.000112         3     FAC-19   
3  1.525360e+06  601560.000571         4     FAC-12   
4  1.556115e+06  626720.964439         5      FAC-4   

                                              NAME  SUBTYPEFIELD  \
0  Bakers Creek Park/Greenway - City of Kannapolis           820   
1               Frank Liske Park - Cabarrus County           820   
2                                  McAllister Park           820   
3   Hartsell Park and Rec Center - City of Concord           820   
4                   Camp Spencer - Cabarrus County           820   

      FEATURECODE                             FULLADDR   OPERDAYS  \
0  Municipal Park         1275 West A St, 704-920-4343  Sun - Sat   
1     County Park         4001 Stough Rd, 704-920-3350  Sun - Sat   
2            Park                         8595 Park D

In [3]:
# ============================================================
# STEP 2 — SQL QUERY
# Pull only the columns we need from the database
# WHERE X IS NOT NULL AND Y IS NOT NULL makes sure
# every park has coordinates — no coordinates = can't map it
# ============================================================
query = """
SELECT NAME, FULLADDR, FEATURECODE,
       PLAYGROUND, BASEBALL, BASKETBALL,
       SOCCER, HIKING, PICNIC, FISHING,
       BOATING, RESTROOM, ADACOMPLY,
       SWIMMING, CAMPING, GOLF,
       PARKAREA, X, Y
FROM parks
WHERE X IS NOT NULL
AND Y IS NOT NULL
"""

# pd.read_sql_query runs the SQL and stores results in df
# df is now a Pandas DataFrame — a table in memory
df = pd.read_sql_query(query, conn)

print(f"Parks loaded: {len(df)}")

print("\nPREVIEW OF FIRST 3 PARKS")
print(df.head(3).T)  


Parks loaded: 57

PREVIEW OF FIRST 3 PARKS
                                                           0  \
NAME         Bakers Creek Park/Greenway - City of Kannapolis   
FULLADDR                        1275 West A St, 704-920-4343   
FEATURECODE                                   Municipal Park   
PLAYGROUND                                               Yes   
BASEBALL                                                 Yes   
BASKETBALL                                               Yes   
SOCCER                                                    No   
HIKING                                                   Yes   
PICNIC                                                   Yes   
FISHING                                                   No   
BOATING                                                   No   
RESTROOM                                                 Yes   
ADACOMPLY                                                Yes   
SWIMMING                                                  No 

In [4]:
# ============================================================
# STEP 3 — CREATE NEW COLUMNS FROM EXISTING DATA
# SQL gave us raw Yes/No data — Pandas calculates scores
# ============================================================

# Plain Python list of column names we want to count
# Stored as variable so we don't retype them multiple times
amenity_cols = ['PLAYGROUND','BASEBALL','BASKETBALL',
                'SOCCER','HIKING','PICNIC',
                'FISHING','BOATING','SWIMMING',
                'CAMPING','GOLF']

# ---- CRITICAL: LEFT vs RIGHT SIDE OF = ----
# df['col'] = something   → LEFT  = CREATE/OVERWRITE column
# x = df['col'].mean()    → RIGHT = READ column, do math
# Same syntax, different job depending on side of =

# ---- HOW AMENITY_SCORE IS CALCULATED ----
# df[col] == 'Yes'     → True or False per row
# .astype(int)         → True→1, False→0
# sum(for each col)    → adds all 1s per row = final score
# Frank Liske has 8 amenities → score of 8
df['AMENITY_SCORE'] = sum(
    (df[col] == 'Yes').astype(int) for col in amenity_cols
)

# Double bracket = table format with headers (use for display)
# .to_string() = show ALL rows, no ... cutoff
print("\nPARK AMENITY SCORES")
print(df[['NAME', 'AMENITY_SCORE']].to_string())

# Single bracket = plain number (use for math)
avg     = df['AMENITY_SCORE'].mean()  # average
highest = df['AMENITY_SCORE'].max()   # highest value
lowest  = df['AMENITY_SCORE'].min()   # lowest value

# :.2f = round to 2 decimal places
print(f"\nAverage amenity score: {avg:.2f}")
print(f"Highest scoring park:  {highest}")
print(f"Lowest scoring park:   {lowest}")

# ============================================================
# PANDAS QUICK REFERENCE
#
# DISPLAY:
# df.head(n)              first n rows
# df.tail(n)              last n rows
# df.shape                (rows, columns)
# df.columns              all column names
# df.dtypes               data type per column
# df.isnull().sum()       null count per column
# df.describe()           stats: mean, min, max, std
#
# MATH ON A COLUMN (single bracket):
# df['col'].mean()        average
# df['col'].max()         highest
# df['col'].min()         lowest
# df['col'].sum()         total
# df['col'].count()       non-null count
# df['col'].nunique()     unique value count
# df['col'].value_counts() count of each unique value
#
# CREATE NEW COLUMN (left side of =):
# df['new'] = 5                              same value all rows
# df['new'] = df['a'] + df['b']             math between columns
# df['new'] = (df['col'] == 'Yes').astype(int)  Yes/No to 1/0
#
# FILTER ROWS:
# df[df['col'] == 'Yes']      where col equals Yes
# df[df['col'] > 5]           where col greater than 5
# df[df['col'].isnull()]      where col is empty
# df[df['col'].notnull()]     where col is not empty
#
# BRACKETS:
# df['col']            single = Series, use for math
# df[['col']]          double = DataFrame, use to display
# df[['col1','col2']]  double = multiple columns as table
# ============================================================


PARK AMENITY SCORES
                                               NAME  AMENITY_SCORE
0   Bakers Creek Park/Greenway - City of Kannapolis              6
1                Frank Liske Park - Cabarrus County              8
2                                   McAllister Park              2
3    Hartsell Park and Rec Center - City of Concord              5
4                    Camp Spencer - Cabarrus County              6
5                Veterans Park - City of Kannapolis              0
6              Beverly Hills Park - City of Concord              2
7                 Village Park - City of Kannapolis              2
8                 W.W. Flowe Park - City of Concord              4
9                       Beach Springs Mt. Bike Park              0
10        Pharr Mill Road Park - Town of Harrisburg              4
11        Harrisburg Town Park - Town of Harrisburg              4
12         Stallings Road Park - Town of Harrisburg              4
13                       AT Allen Element

In [5]:
# ============================================================
# STEP 4 — ADD SIZE_CAT AND ADA_LABEL COLUMNS
# np.select = same as CASE WHEN in SQL
# conditions = list of if statements checked top to bottom
# choices = what to assign when condition is true
# default = what to use when none of the conditions match
# (covers parks where PARKAREA is null = Unknown)
# ============================================================

conditions = [
    df['PARKAREA'] < 10,
    (df['PARKAREA'] >= 10) & (df['PARKAREA'] < 50),
    (df['PARKAREA'] >= 50) & (df['PARKAREA'] < 200),
    df['PARKAREA'] >= 200
]
choices = ['Small', 'Medium', 'Large', 'Regional']

df['SIZE_CAT'] = np.select(conditions, choices, default='Unknown')

# np.where = same as a simple IF/ELSE
# if ADACOMPLY = Yes → 'ADA Compliant'
# anything else      → 'Not ADA Compliant'
df['ADA_LABEL'] = np.where(
    df['ADACOMPLY'] == 'Yes',
    'ADA Compliant',
    'Not ADA Compliant'
)

# Verify new columns exist and look right
print("NEW COLUMNS ADDED:")
print(df[['NAME', 'AMENITY_SCORE', 'SIZE_CAT', 'ADA_LABEL']].to_string())

NEW COLUMNS ADDED:
                                               NAME  AMENITY_SCORE  SIZE_CAT          ADA_LABEL
0   Bakers Creek Park/Greenway - City of Kannapolis              6    Medium      ADA Compliant
1                Frank Liske Park - Cabarrus County              8  Regional      ADA Compliant
2                                   McAllister Park              2   Unknown      ADA Compliant
3    Hartsell Park and Rec Center - City of Concord              5    Medium      ADA Compliant
4                    Camp Spencer - Cabarrus County              6     Large      ADA Compliant
5                Veterans Park - City of Kannapolis              0   Unknown      ADA Compliant
6              Beverly Hills Park - City of Concord              2     Small      ADA Compliant
7                 Village Park - City of Kannapolis              2   Unknown      ADA Compliant
8                 W.W. Flowe Park - City of Concord              4   Unknown      ADA Compliant
9                    

In [10]:
# ============================================================
# STEP 5 — CONVERT TO GEOPANDAS + EXPORT TO GEOJSON
# ============================================================

# ---- CRITICAL: HOW TO FIND THE RIGHT CRS ----
#
# Every spatial dataset has a CRS (Coordinate Reference System)
# It tells GIS software what the X Y numbers mean
# Using the wrong CRS = dots in the ocean instead of NC
#
# STEP 1 — CHECK YOUR COORDINATE RANGES
# print(f"X range: {df['X'].min():.2f} to {df['X'].max():.2f}")
# print(f"Y range: {df['Y'].min():.2f} to {df['Y'].max():.2f}")
#
# STEP 2 — READ THE NUMBERS
# If X is around -180 to 180 AND Y is around -90 to 90
#   → already lat/lon → use EPSG:4326, no reprojection needed
#
# If X is large numbers like 1,400,000 to 1,600,000
# and Y is large numbers like 500,000 to 700,000
#   → projected coordinates in feet or meters
#   → need to figure out which one
#
# STEP 3 — TEST WHICH CRS IS CORRECT
# Use pyproj to convert a known point and check the result:
#
# import pyproj
# x = df['X'].iloc[0]   # grab first X value
# y = df['Y'].iloc[0]   # grab first Y value
#
# # Test a CRS — transform to lat/lon and check if it looks right
# transformer = pyproj.Transformer.from_crs("EPSG:2264", "EPSG:4326", always_xy=True)
# lon, lat = transformer.transform(x, y)
# print(f"lon={lon:.4f}, lat={lat:.4f}")
# # NC should be around lon=-80.5, lat=35.4
# # If it looks right — that CRS is correct
# # If it's wrong — try a different EPSG code
#
# COMMON NC CRS OPTIONS TO TRY:
# EPSG:4326  = WGS84 lat/lon       (degrees, global standard)
# EPSG:32119 = NC State Plane      (meters)
# EPSG:2264  = NC State Plane      (US survey feet) ← this data
# EPSG:3857  = Web Mercator        (meters, used by Google Maps)
#
# THIS DATASET:
# X range 1,481,128 to 1,573,466 = NC State Plane US Survey Feet
# Correct CRS = EPSG:2264
# Confirmed by testing: EPSG:2264 → lon=-80.62, lat=35.50 ✓
# ============================================================

# EPSG:2264 = NAD83 NC State Plane US Survey Feet
# This matches the X Y values from the Cabarrus parks CSV
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.X, df.Y),
    crs="EPSG:2264"
)

print(f"GeoDataFrame created: {len(gdf)} parks")
print(f"CRS: {gdf.crs}")

# Reproject to WGS84 lat/lon
# AGOL and most web mapping tools require EPSG:4326
# .to_crs() converts coordinates from one system to another
# After this geometry column contains lat/lon degrees
gdf = gdf.to_crs("EPSG:4326")
print(f"Reprojected to: {gdf.crs}")

# ---- VERIFY BEFORE EXPORTING ----
# Always check one coordinate after reprojecting
# If it looks like NC (-80.x, 35.x) you're good
# If it looks wrong — stop and fix the CRS before exporting
sample = gdf.geometry.iloc[0]
print(f"\nSample coordinate check:")
print(f"lon={sample.x:.4f}, lat={sample.y:.4f}")
print(f"Should be around lon=-80.5, lat=35.4 for Cabarrus NC")

# Export to GeoJSON
# GeoJSON = web friendly spatial format
# Drag and drop directly into AGOL to upload
# driver='GeoJSON' tells GeoPandas what file format to write
output_path = r"C:\cabarrus-gis-prep\geopandas\outputs\cabarrus_parks_web.geojson"

gdf.drop(columns=['X', 'Y'], errors='ignore').to_file(
    output_path,
    driver='GeoJSON'
)

print(f"\nExported successfully")
print(f"File: {output_path}")
print(f"Parks exported: {len(gdf)}")

GeoDataFrame created: 57 parks
CRS: EPSG:2264
Reprojected to: EPSG:4326

Sample coordinate check:
lon=-80.6292, lat=35.5089
Should be around lon=-80.5, lat=35.4 for Cabarrus NC

Exported successfully
File: C:\cabarrus-gis-prep\geopandas\outputs\cabarrus_parks_web.geojson
Parks exported: 57


In [ ]:
# These are your actual coordinate ranges
# X: 1481128 to 1573466
# Y: 543252  to 644107
# Let's test the correct NC CRS options

import pyproj

# Test point — Bakers Creek Park
x = 1515174.499324
y = 644107.563114

# Option 1 — NC State Plane feet (what we tried)
crs1 = pyproj.CRS("EPSG:32119")
print(f"EPSG:32119 name: {crs1.name}")

# Option 2 — NC State Plane US Survey Feet
crs2 = pyproj.CRS("EPSG:2264")
print(f"EPSG:2264 name: {crs2.name}")

# Convert using option 2 and see if it gives NC coordinates
transformer = pyproj.Transformer.from_crs("EPSG:2264", "EPSG:4326", always_xy=True)
lon, lat = transformer.transform(x, y)
print(f"\nEPSG:2264 result: lon={lon:.4f}, lat={lat:.4f}")
print(f"Should be around: lon=-80.5, lat=35.4")

EPSG:32119 name: NAD83 / North Carolina
EPSG:2264 name: NAD83 / North Carolina (ftUS)

EPSG:2264 result: lon=-80.6292, lat=35.5089
Should be around: lon=-80.5, lat=35.4
